# Chapter 21: The AI Agent

Natural language interface to PyGeoVision's complete geospatial pipeline.

In [ ]:
import sys; sys.path.insert(0,"../..")
import pygeovision as pgv
from pygeovision.agent import GeoAgent, HeuristicPlanner
from pygeovision.agent.tools import build_tools, TOOL_REGISTRY
from unittest.mock import MagicMock
print(f"GeoAgent tools: {list(TOOL_REGISTRY.keys())}")

## Decision matrix demo

In [ ]:
tools   = build_tools(MagicMock())
planner = HeuristicPlanner(tools)
ctx     = {"bbox":[-0.3,5.5,-0.05,5.7],"date":"2026-06","output_dir":"/tmp/"}

queries = [
    "Map SAR flood extent — cloud cover 100%",
    "Detect building damage after earthquake using Sentinel-1",
    "Map flood using Sentinel-2 optical imagery",
    "Classify land cover using Prithvi foundation model",
    "Compute NDVI vegetation index",
    "Monitor land subsidence in Jakarta",
    "Detect oil spill with Sentinel-1",
    "Extract building footprints in Dubai",
]

print(f"{'Query':<50} {'Sensor':>8}  {'Task':>15}")
print("-"*80)
for q in queries:
    p = planner.plan(q, context=ctx)
    print(f"{q[:49]:<50} {p.sensor:>8}  {p.task:>15}")

## Session and memory

In [ ]:
from pygeovision.agent.memory import GeoAgentMemory

mem = GeoAgentMemory()
mem.set_context(bbox=(-0.30, 5.50, -0.05, 5.70), date="2026-06",
                 output_dir="./accra_analysis/")
mem.bind("flood_mask", "./results/flood.geojson")

print(mem)
print("\nContext for planner:")
print(mem.context_for_planner())

## Custom tool example

In [ ]:
from pygeovision.agent.tools import GeoTool, ToolResult, TOOL_REGISTRY

class DemoAnalysisTool(GeoTool):
    name        = "demo_analysis"
    description = "Compute area statistics from a raster mask"
    parameters  = [{"name":"mask_path","type":"str","required":True},
                    {"name":"pixel_size_m","type":"float","required":False}]

    def run(self, mask_path, pixel_size_m=10, **_):
        import numpy as np
        try:
            import rasterio
            with rasterio.open(mask_path) as src:
                mask = src.read(1); pix = abs(src.transform.a)
        except Exception:
            mask = np.zeros((64,64)); pix = pixel_size_m
        area_km2 = float((mask > 0).sum()) * pix**2 / 1e6
        return ToolResult(tool=self.name, success=True,
                           output={"area_km2": area_km2})

TOOL_REGISTRY["demo_analysis"] = DemoAnalysisTool
print(f"Custom tool registered. Total tools: {len(TOOL_REGISTRY)}")

## Streaming simulation

In [ ]:
import time

def simulate_stream(query, n_steps=3):
    tools   = build_tools(MagicMock())
    planner = HeuristicPlanner(tools)
    plan    = planner.plan(query, context=ctx)

    yield {"type":"plan", "steps":len(plan.steps), "planner":"heuristic",
            "sensor":plan.sensor, "task":plan.task}

    for i, step in enumerate(plan.steps):
        yield {"type":"step_start", "step":i, "tool":step.tool_name}
        time.sleep(0.05)  # simulate execution
        yield {"type":"step_done", "step":i, "tool":step.tool_name,
                "success":True, "duration_s":0.05}

    yield {"type":"execution_complete", "success":True,
            "final_output":"/tmp/result.geojson"}

query = "Map flood inundation using SAR — monsoon season, heavy cloud cover"
print(f"Query: {query}\n")
for event in simulate_stream(query):
    t = event["type"]
    if t == "plan":
        print(f"Plan: {event['steps']} steps | sensor={event['sensor']} task={event['task']}")
    elif t == "step_start":
        print(f"  [{event['step']}] > {event['tool']}...")
    elif t == "step_done":
        print(f"  [{event['step']}] OK  {event['tool']} ({event['duration_s']:.2f}s)")
    elif t == "execution_complete":
        print(f"\nDone: {event['final_output']}")